<a href="https://colab.research.google.com/github/Zarrialvi/AI-PROJECTS/blob/main/GDP_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # GDP Prediction Model (2020-2025)
#
# This notebook walks through the process of downloading the "GDP per Country 2020-2025" dataset from Kaggle, performing exploratory data analysis, preprocessing the data, and training a machine learning model to predict future GDP.

# %% [markdown]
# ## 1. Setup and Authentication
#
# First, we need to set up the Kaggle API to download the dataset directly.
#
# 1.  Go to your Kaggle account, click on your profile picture, and select "Account".
# 2.  Scroll down to the "API" section and click "Create New API Token". This will download a `kaggle.json` file.
# 3.  Upload the `kaggle.json` file to your Colab environment using the code cell below.

# %%
# Import necessary libraries for file handling
import os
from google.colab import files

# Prompt to upload the kaggle.json file
print("Please upload your kaggle.json file")
files.upload()

# Create a directory for the Kaggle API credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\nKaggle API configured successfully!")

# %% [markdown]
# ## 2. Download and Prepare the Dataset
#
# Now we'll use the Kaggle API to download the dataset and unzip it.

# %%
# Download the dataset using the Kaggle API
!kaggle datasets download -d codebynadiia/gdp-per-country-20202025

# Unzip the downloaded file
!unzip gdp-per-country-20202025.zip

print("\nDataset downloaded and unzipped.")
# List files to find the CSV name
!ls

# %% [markdown]
# ## 3. Load and Explore the Data (EDA)
#
# We will load the data into a pandas DataFrame and perform some basic exploratory data analysis to understand its structure and content.

# %%
# Import data analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
# Let's assume the CSV file is named 'gdp_per_capita.csv'.
# Please adjust the filename if it's different after running the cell above.
try:
    df = pd.read_csv('gdp_per_capita.csv')
except FileNotFoundError:
    print("Error: The CSV file 'gdp_per_capita.csv' was not found.")
    print("Please check the file name from the output of the previous cell and update the code.")
    df = None

if df is not None:
    # Display the first few rows of the dataframe
    print("First 5 rows of the dataset:")
    display(df.head())

    # Get a summary of the dataframe's structure and data types
    print("\nDataset Info:")
    df.info()

    # Get descriptive statistics for the numerical columns
    print("\nDescriptive Statistics:")
    display(df.describe())

    # Check for any missing values
    print("\nMissing Values:")
    print(df.isnull().sum())


# %% [markdown]
# ### 3.1. Advanced Exploratory Data Analysis
#
# Let's dive deeper into the data with some visualizations.

# %% [markdown]
# #### GDP Distribution (2024)
#
# We'll look at the distribution of GDP per capita for the year 2024. This will help us understand the spread, central tendency, and identify any potential outliers. Since GDP data is often skewed, we'll also look at the log-transformed distribution.

# %%
if df is not None:
    # Set the style for plots
    sns.set_style("whitegrid")

    # Create a figure for the plots
    plt.figure(figsize=(16, 6))

    # Plot the distribution of GDP for 2024
    plt.subplot(1, 2, 1)
    sns.histplot(df['2024'], kde=True, bins=30)
    plt.title('Distribution of GDP per Capita for 2024', fontsize=16)
    plt.xlabel('GDP (2024)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    # Plot the log-transformed distribution of GDP for 2024
    plt.subplot(1, 2, 2)
    sns.histplot(np.log1p(df['2024']), kde=True, bins=30, color='salmon')
    plt.title('Log-Transformed Distribution of GDP per Capita for 2024', fontsize=16)
    plt.xlabel('Log(1 + GDP (2024))', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    plt.tight_layout()
    plt.show()


# %% [markdown]
# #### Top 15 Countries by GDP (2024)
#
# Let's identify the countries with the highest GDP per capita in 2024.

# %%
if df is not None:
    # Get the top 15 countries by 2024 GDP
    top_15_gdp = df.sort_values(by='2024', ascending=False).head(15)

    plt.figure(figsize=(12, 8))
    sns.barplot(x='2024', y='Country Name', data=top_15_gdp, palette='viridis')
    plt.title('Top 15 Countries by GDP per Capita (2024)', fontsize=16)
    plt.xlabel('GDP per Capita (2024)', fontsize=12)
    plt.ylabel('Country', fontsize=12)
    plt.show()


# %% [markdown]
# #### Correlation Between Years
#
# A correlation heatmap will show us how strongly the GDP in one year is related to the GDP in another. We expect to see a very high correlation, as a country's GDP doesn't typically change dramatically year-over-year.

# %%
if df is not None:
    # Select only the year columns for the correlation matrix
    year_columns = ['2020', '2021', '2022', '2023', '2024', '2025']
    correlation_matrix = df[year_columns].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=.5)
    plt.title('Correlation Heatmap of GDP Across Years', fontsize=16)
    plt.show()


# %% [markdown]
# #### GDP Growth Trends for Selected Countries
#
# Let's plot the GDP per capita over the years for a few selected countries to visualize their growth trends. We'll pick some of the top countries and a few others for comparison.

# %%
if df is not None:
    # Reshape the data from wide to long format for easier plotting
    df_long = df.melt(id_vars=['Country Name', 'Country Code'],
                      var_name='Year',
                      value_name='GDP')
    df_long['Year'] = pd.to_numeric(df_long['Year'])

    # Select some countries to plot
    countries_to_plot = ['United States', 'China', 'Japan', 'Germany', 'India', 'United Kingdom', 'Nigeria']
    df_plot = df_long[df_long['Country Name'].isin(countries_to_plot)]

    plt.figure(figsize=(14, 8))
    sns.lineplot(x='Year', y='GDP', hue='Country Name', data=df_plot, marker='o', lw=2.5)
    plt.title('GDP Per Capita Growth (2020-2025)', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('GDP per Capita', fontsize=12)
    plt.legend(title='Country')
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

# %% [markdown]
# ## 4. Data Preprocessing
#
# Before training a model, we need to prepare the data. This involves:
# 1.  Handling categorical data ('Country Name').
# 2.  Separating features (X) from the target variable (y). We will predict the GDP for 2025.
# 3.  Splitting the data into training and testing sets.
# 4.  Scaling the numerical features to ensure they have a similar range.

# %%
if df is not None:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder

    # Drop 'Country Code' as it's redundant with 'Country Name'
    df_processed = df.drop('Country Code', axis=1)

    # Define features (X) and target (y)
    # We will use GDP data from 2020 to 2024 to predict 2025 GDP.
    X = df_processed.drop(['2025'], axis=1)
    y = df_processed['2025']

    # Identify categorical and numerical features
    categorical_features = ['Country Name']
    numerical_features = X.columns.drop(categorical_features)

    # Create a preprocessor to handle both categorical and numerical data
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Data preprocessing complete.")
    print(f"Training set size: {X_train.shape[0]} samples")
    print(f"Testing set size: {X_test.shape[0]} samples")


# %% [markdown]
# ## 5. Train a Machine Learning Model
#
# We will use a `RandomForestRegressor`, which is a powerful and versatile model suitable for this kind of regression task. We'll wrap our preprocessor and model in a `Pipeline` to streamline the process.

# %%
if df is not None:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    # Create the model pipeline
    model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

    # Train the model
    print("Training the RandomForestRegressor model...")
    model.fit(X_train, y_train)
    print("Model training complete.")

    # Make predictions on the test set
    y_pred = model.predict(X_test)


# %% [markdown]
# ## 6. Evaluate the Model
#
# Let's evaluate our model's performance on the test data using common regression metrics: Mean Absolute Error (MAE), Mean Squared Error (MSE), and R-squared (R²).

# %%
if df is not None:
    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("Model Evaluation Metrics:")
    print(f"  Mean Absolute Error (MAE): {mae:.2f}")
    print(f"  Mean Squared Error (MSE): {mse:.2f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"  R-squared (R²): {r2:.4f}")

    # Visualize the results: Actual vs. Predicted values
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.7, edgecolors='w', linewidth=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='red', lw=2)
    plt.title('Actual vs. Predicted GDP for 2025', fontsize=16)
    plt.xlabel('Actual GDP (2025)', fontsize=12)
    plt.ylabel('Predicted GDP (2025)', fontsize=12)
    plt.grid(True)
    plt.show()

Please upload your kaggle.json file


Saving kaggle.json to kaggle.json

Kaggle API configured successfully!
Dataset URL: https://www.kaggle.com/datasets/codebynadiia/gdp-per-country-20202025
License(s): MIT
  0% 0.00/5.54k [00:00<?, ?B/s]
100% 5.54k/5.54k [00:00<00:00, 22.5MB/s]
Archive:  gdp-per-country-20202025.zip
  inflating: 2020-2025.csv           

Dataset downloaded and unzipped.
2020-2025.csv  gdp-per-country-20202025.zip  kaggle.json  sample_data
Error: The CSV file 'gdp_per_capita.csv' was not found.
Please check the file name from the output of the previous cell and update the code.
